In [4]:
import glob
import re
from pathlib import Path

from natsort import natsorted

# --- CONFIGURATION ---

# The base directory containing the 'mouse_i' folders for Replicate 1.
# NOTE: You MUST update this path to the correct mount point on your Ubuntu desktop.
BASE_DIR = Path("/Volumes/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice")

# Fixed experimental conditions for Replicate 1 (Mouse 1 through 12)
REP1_CONDITIONS = {
    1: 'H2O', 2: 'DMSO', 3: 'RIF', 4: 'RIF', 
    5: 'RIF', 6: 'RIF', 7: 'PZA', 8: 'PZA', 
    9: 'PZA', 10: 'PZA', 11: 'DMSO', 12: 'DMSO'
}

# --- EXECUTION ---

# Dictionary to store the final mapping: 
# {Zarr_Basename_Stem: {'mouse_num': X, 'replicate': 1, 'condition': C}}
final_metadata_map = {}

print(f"Scanning directory structure: {BASE_DIR}")
print("---")

# 1. Find all mouse directories (e.g., 'mouse_1', 'mouse_2', etc.)
# We use natsorted for correct numerical ordering (mouse_1, mouse_2, ..., mouse_12)
mouse_dirs = natsorted(glob.glob(str(BASE_DIR / "mouse_*")))

for mouse_path_str in mouse_dirs:
    mouse_path = Path(mouse_path_str)
    
    # 2. Extract Mouse Number and filter for Replicate 1 (Mouse 1-12)
    match = re.search(r'mouse_(\d+)', mouse_path.name)
    if not match:
        continue
    
    mouse_num = int(match.group(1))

    if mouse_num not in REP1_CONDITIONS:
        # Stop processing once we go beyond the range of Replicate 1 mice
        continue

    # 3. Find Zarr file(s) within the mouse_i/zarr/ subdirectory
    # Looks for any file ending in .zarr inside the 'zarr' subdirectory
    zarr_files = list(mouse_path.glob("zarr/*.zarr"))

    if not zarr_files:
        print(f"Warning: No Zarr file found in {mouse_path.name}/zarr/")
        continue

    # 4. Process the found Zarr file
    # Assuming one Zarr file per mouse folder for this analysis
    for zarr_fn in zarr_files:
        if 'corrupted' in zarr_fn.name: 
            print(f"  Skipping corrupted file: {zarr_fn.name}")
            continue 
        if 'notebook' in zarr_fn.name: 
            print(f"  Skipping corrupted file: {zarr_fn.name}")
            continue 
        basename = zarr_fn.stem # Get the filename stem (basename without '.zarr')
        
        # Look up the condition using the extracted mouse number
        condition_str = REP1_CONDITIONS[mouse_num]
        
        # Populate the final dictionary
        final_metadata_map[basename] = {
            'mouse_num': mouse_num,
            'replicate': 1,
            'condition': condition_str
        }
        
        print(f"  Mapped {mouse_path.name} -> {zarr_fn.name} (Condition: {condition_str})")

print("\n--- Final Replicate 1 Metadata Map ---")
# Print the results in a nicely readable format (Basename as Key)
for basename, data in final_metadata_map.items():
    print(f"'{basename}': {data}")

print(f"\n✅ Total entries generated: {len(final_metadata_map)}")

Scanning directory structure: /Volumes/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice
---
  Mapped mouse_1 -> 20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250815_5375.zarr (Condition: H2O)
  Skipping corrupted file: 20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5551_jnotebook.zarr
  Mapped mouse_2 -> 20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5551.zarr (Condition: DMSO)
  Skipping corrupted file: 20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5552_jnotebook.zarr
  Mapped mouse_3 -> 20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5552.zarr (Condition: RIF)
  Mapped mouse_4 -> 20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5553.zarr (Condition: RIF)
  Skipping corrupted file: 20250901_40X_TimerMtb_BP_mice2_mice3_mice4_

In [6]:
import pandas as pd

In [7]:

# Convert the dictionary where keys are the index (Zarr basename)
df_metadata = pd.DataFrame.from_dict(final_metadata_map, orient='index')

# Rename the index column to be explicit
df_metadata = df_metadata.rename_axis('zarr_basename_stem').reset_index()

In [11]:
pd.set_option('display.max_rows', None)        # Show all rows
pd.set_option('display.max_columns', None)     # Show all columns
pd.set_option('display.width', 1000)           # Adjust console width
pd.set_option('display.max_colwidth', None) 

In [12]:
df_metadata

,zarr_basename_stem,mouse_num,replicate,condition
0,20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250815_5375,1,1,H2O
1,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5551,2,1,DMSO
2,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5552,3,1,RIF
3,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5553,4,1,RIF
4,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5555,5,1,RIF
5,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556,6,1,RIF
6,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557,7,1,PZA
7,20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549,8,1,PZA
8,20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550,9,1,PZA
9,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375,10,1,PZA


In [14]:
import glob
import re
from pathlib import Path

import pandas as pd
from natsort import natsorted

# --- NEW CODE: SET PANDAS DISPLAY OPTIONS ---
pd.set_option('display.max_rows', None)        # Show all rows
pd.set_option('display.max_columns', None)     # Show all columns
pd.set_option('display.width', 1000)           # Adjust console width
pd.set_option('display.max_colwidth', None)    # Show full column width (no truncation)
# -------------------------------------------


# --- CONFIGURATION ---

# The base directory containing the 'mouse_i' folders for Replicate 2.
# NOTE: You MUST update this path to the correct mount point on your Ubuntu desktop.
BASE_DIR = Path("/Volumes/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/rep2")

# Fixed experimental conditions for Replicate 2 (Mouse 1 through 16)
# 1-3 = water control group
# 4-6 = DMSO control group
# 7-11 = RIF treated
# 12-16 = PZA treated
REP2_CONDITIONS_MAP = {
    range(1, 4): 'H2O',          # Mouse 1, 2, 3
    range(4, 7): 'DMSO',         # Mouse 4, 5, 6
    range(7, 12): 'RIF',         # Mouse 7, 8, 9, 10, 11
    range(12, 17): 'PZA'         # Mouse 12, 13, 14, 15, 16
}

def get_condition_by_mouse(mouse_num):
    """Determines the condition string based on the mouse number."""
    for mouse_range, condition in REP2_CONDITIONS_MAP.items():
        if mouse_num in mouse_range:
            return condition
    return 'UNKNOWN'

# --- EXECUTION ---

# Dictionary to store the final mapping: 
# {VSI_Basename_Stem: {'mouse_num': X, 'replicate': 2, 'condition': C}}
final_metadata_map = {}

print(f"Scanning directory structure: {BASE_DIR} for Replicate 2 VSI files...")
print("---")

# 1. Find all mouse directories for Replicate 2 (up to mouse_16)
mouse_dirs = natsorted(glob.glob(str(BASE_DIR / "mouse_*")))

for mouse_path_str in mouse_dirs:
    mouse_path = Path(mouse_path_str)
    
    # 2. Extract Mouse Number
    match = re.search(r'mouse_(\d+)', mouse_path.name)
    if not match:
        continue
    
    mouse_num = int(match.group(1))

    if mouse_num > 16:
        # Stop processing if we go beyond the scope of Rep2
        continue

    # 3. Find VSI file(s) within the mouse_i/vsi/ subdirectory
    # We are looking for any file ending in .vsi
    vsi_files = list(mouse_path.glob("vsi/*.vsi"))

    if not vsi_files:
        print(f"  Warning: No VSI file found in {mouse_path.name}/vsi/")
        continue

    # 4. Process the found VSI file(s)
    for vsi_fn in vsi_files:
        # Skip the temporary memo files or other related non-primary files
        if vsi_fn.name.startswith('.'): 
            continue
        
        basename = vsi_fn.stem # Get the filename stem (basename without '.vsi')
        
        # Look up the condition using the mouse number
        condition_str = get_condition_by_mouse(mouse_num)
        
        # Populate the final dictionary
        final_metadata_map[basename] = {
            'mouse_num': mouse_num,
            'replicate': 2,
            'condition': condition_str
        }
        
        print(f"  Mapped {mouse_path.name} -> {vsi_fn.name} (Condition: {condition_str})")

print("\n--- Final Replicate 2 Metadata Map ---")

# --- CONVERT TO DATAFRAME ---

# Convert the dictionary where keys are the index (VSI basename)
df_rep2_metadata = pd.DataFrame.from_dict(final_metadata_map, orient='index')

# Rename the index column to be explicit
df_rep2_metadata = df_rep2_metadata.rename_axis('vsi_basename_stem').reset_index()

df_rep2_metadata

Scanning directory structure: /Volumes/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/rep2 for Replicate 2 VSI files...
---
  Mapped mouse_1 -> 20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5574.vsi (Condition: H2O)
  Mapped mouse_2 -> 20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5573.vsi (Condition: H2O)
  Mapped mouse_3 -> 20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5572.vsi (Condition: H2O)
  Mapped mouse_4 -> 20251014_40X_TimerMtb_BP_rep2_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251015_5587.vsi (Condition: DMSO)
  Mapped mouse_5 -> 20251014_40X_TimerMtb_BP_rep2_mice5_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251015_5589.vsi (Condition: DMSO)
  Mapped mouse_6 -> 20251014_40X_TimerMtb_BP_rep2_mice6_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251016_5590.vsi (Condition: DMSO)
  Mapped mouse_7 -> 20251007_40X_T

,vsi_basename_stem,mouse_num,replicate,condition
0,20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5574,1,2,H2O
1,20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5573,2,2,H2O
2,20251001_40X_TimerMtb_BP_rep2_mice3_mice2_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251002_5572,3,2,H2O
3,20251014_40X_TimerMtb_BP_rep2_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251015_5587,4,2,DMSO
4,20251014_40X_TimerMtb_BP_rep2_mice5_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251015_5589,5,2,DMSO
5,20251014_40X_TimerMtb_BP_rep2_mice6_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251016_5590,6,2,DMSO
6,20251007_40X_TimerMtb_BP_rep2_mice7_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251008_5584,7,2,RIF
7,20251007_40X_TimerMtb_BP_rep2_mice7_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251008_5585,8,2,RIF
8,20251007_40X_TimerMtb_BP_rep2_mice7_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251008_5586,9,2,RIF
9,20251030_40X_TimerMtb_BP_rep2_mice11_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20251106_5691,10,2,RIF


In [16]:
df = pd.concat([df_metadata,df_rep2_metadata])

In [17]:
df['basename_stem'] = df['zarr_basename_stem'].combine_first(df['vsi_basename_stem'])

# Remove the two source columns, as their data is now unified in 'basename_stem'
df = df.drop(columns=['zarr_basename_stem', 'vsi_basename_stem'])

# Display the result to confirm the structure
print("\n--- Unified Combined Metadata DataFrame ---")
print(df.head())


--- Unified Combined Metadata DataFrame ---
   mouse_num  replicate condition                                                                                     basename_stem
0          1          1       H2O              20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250815_5375
1          2          1      DMSO  20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5551
2          3          1       RIF  20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5552
3          4          1       RIF  20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5553
4          5          1       RIF  20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5555


In [18]:
df

,mouse_num,replicate,condition,basename_stem
0,1,1,H2O,20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250815_5375
1,2,1,DMSO,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5551
2,3,1,RIF,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5552
3,4,1,RIF,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250901_5553
4,5,1,RIF,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5555
5,6,1,RIF,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5556
6,7,1,PZA,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557
7,8,1,PZA,20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549
8,9,1,PZA,20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550
9,10,1,PZA,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375


In [19]:
df.to_pickle('/Volumes/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/metadata.pkl')